<a href="https://colab.research.google.com/github/takatakamanbou/ML/blob/2025/ML2025_ex07notebookB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML ex07notebookB

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/ML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?ML/2025)


----
## 事前学習済みモデルの利用
----






---
### 転移学習とファインチューニング

近年のいわゆる人工知能（AI, Airtificial Intelligence）のシステムの裏には，ニューラルネットワークをはじめとする様々な機械学習の技術があります．難しい課題をこなせるようなシステムを作るためには，学習データを大量に用意して大規模な（パラメータ数が多く複雑な）モデルを学習させることが必要なわけですが，そのような学習をすべて自前でやろうとすると，多大な手間と時間がかかります．

このような手間と時間を軽減する方法として，画像認識，音声認識，自然言語処理等の汎用性の高い問題を解くような学習モデルを，大量のデータを用いて事前に学習させておき，得られたモデルを個別の問題ごとに調整して使う，というやり方がよく用いられます．

このようなやり方を含めて，ある問題を学習した機械学習モデルを，別の新たな問題にうまく利用できるようにする技術を，**転移学習** (transfer learning) といいます．

例えば，画像を扱う問題をニューラルネットワークで解きたい場合，次のようにすることがよくあります：
1. 大規模な画像データセット（例えば，1000クラスの画像120万枚）を用いて画像識別を学習したニューラルネットワークを作っておく（図左）．このような事前学習済みニューラルネットワークモデルはいろいろ公開されているので，この事前学習の作業を自分でやる必要はない．
1. 事前学習済みニューラルネットワークモデルの一部（通常は入力側の部分）を利用して，自分の問題を解くためのニューラルネットを作る（図右）．このとき，事前学習済みの部分のパラメータは固定して，自分で用意した部分だけ学習させることで，学習の手間を大きく減らすことができる．

<img src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/pretrainedmodel.png">


大規模データで学習した事前学習済みモデルは，データの有用な特徴を抽出する能力に優れているので，上記のようにすることで，事前学習とは多少異なる問題でも優れた性能を発揮し得ます．例えば，画像識別を学習した事前識別モデルは，画像識別に限らず，物体検出や画像のセグメンテーション（画像を画素単位で領域分割する）等の様々な問題に適用されています．


上記の説明では，事前学習済みモデルを新たな問題に適用する際には，そのパラメータ（上の図で「この部分は学習しない」と書いてある部分）は固定して学習しないものとしていました．しかし，場合によっては，事前学習で得られた値を初期値として，新たな問題でこれらのパラメータの値も学習させる（微調整する）ことが有効な場合もあります．ニューラルネットワークを用いる転移学習の方法のうち，このように事前学習済みモデルのパラメータも微調整する方法を，**ファインチューニング** (fine-tuning) といいます．

事前学習済みモデルのパラメータを固定して特徴抽出に用いるだけの方法に比べると，ファインチューニングの方が新たな問題に適用するための学習の手間がかかりますが，より高い汎化性能が得られると期待できます．



----
### 1000種類の画像識別を学習済みのニューラルネットを動かしてみよう

画像認識の事前学習済みニューラルネットワークモデルとしてよく知られた，ResNet-50 というニューラルネットワークの事前学習済みモデルを動かしてみます（notebookC で，このモデルを利用した物体検出の実験を行います）．
ResNet-50 は，アメリカの Microsoft Research の研究チームが開発した ResNet（Residual Network）というニューラルネットワークモデルのうちの，層が 50 層あるものです．
ResNet は，2015年に行われた [ILSVRC2015](https://www.image-net.org/challenges/LSVRC/2015/) という画像識別のコンペティションで世界第1位となった高性能なニューラルネットワークモデルです（注）．
ILSVRC2015では，[ImageNet](https://www.image-net.org/) という大規模な画像データセットの中から選ばれた 1000 種類の物体の画像約 120 万枚を学習データとしていました．

参考文献: [Deep Residual Learning for Image Recognition](https://www.cv-foundation.org/openaccess/content_cvpr_2016/papers/He_Deep_Residual_Learning_CVPR_2016_paper.pdf)

<hr width="50%" align="left">
<span style="font-size: 75%">
※注: ちなみに，この授業でこれまでに何度か登場している VGG16 は，1年前に行われた ILSVRC2014 で世界第2位となったニューラルネットワークモデルで，層の数は16でした．
</span>

---
#### 準備

In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import seaborn
seaborn.set_theme()

import pickle
import torch
from torchvision import models, transforms
from PIL import Image

##### 準備その1: 実験に使う画像データの準備

**注意: 以下の画像をこの実験以外の目的で使用してはいけません**

In [ ]:
!wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/animalphoto.pickle
with open('animalphoto.pickle', 'rb') as f:
    hoge = pickle.load(f)

imgList = []
for d in hoge:
    img = Image.frombytes(**d)
    imgList.append(img)
    fig, ax = plt.subplots(1)
    ax.imshow(img)
    ax.axis('off')
    plt.show()

##### 準備2: 事前学習済みニューラルネットのパラメータの入手

規模の大きいネットワークでパラメータがたくさんありますので，読み込みに少し時間がかかります．

In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
model = models.resnet50(weights=weights)
model.eval()
print(model)

上記のセルを実行すると，次のことが行われます．
1. ResNet50のネットワークモデルを作成する．
1. ImageNetで学習済みのパラメータの値を読み込んでこのネットワークのパラメータに設定する．
1. ネットワークモデルの構造を表示する．

画像を扱うニューラルネットでは，「畳み込み層」(convolutional layer)という特別な構造がよく用いられます．
ResNet50も畳み込み層（上記の出力のうち `Conv2d` というのが畳み込み層）を多数含む階層型ニューラルネットで，層の数は50あります．

##### 準備その3: クラスの番号とクラスラベルの対応表の作成

In [ ]:
labels = weights.meta['categories']
for ik, lab in enumerate(labels):
    if 276 <= ik < 300:
        print(ik, labels[ik])

上記の出力は，1000クラスのうちの一部のものの名前を表します．「ネコ」みたいなものも1クラスではなく，281番 'tabby'（ぶち柄の猫），282番 'tiger_cat'（トラ縞の猫）等々のように分かれています．

---
#### 実験1

上記のサンプル画像たちを ResNet50 に識別させてみましょう．


In [ ]:
#@title サンプル画像の識別
#@markdown 以下で 0 から 3 までの整数を選ぶと4つのサンプル画像の中から一つを選ぶことができます．
i = 1 #@param [0, 1, 2, 3] {type: 'raw'}
img = imgList[i]

# 画像を表示
fig, ax = plt.subplots(1)
ax.imshow(img)
ax.axis('off')
plt.show()

# VGG16 ネットワークに入力して出力を得る
X = torch.unsqueeze(preprocess(imgList[i]), axis=0)
Y = model(X)
Z = torch.nn.functional.softmax(Y, dim=1)
P = Z[0].detach().numpy()

# 出力の値の大きかった方から5つを表示
for i, ik in enumerate(np.argsort(-P)[:5]):
    print(f'rank{i+1}: {P[ik]:.6f} {labels[ik]}')

上記の画像の下の出力は，1000個のクラスの中で，ネットワークの出力の値が大きかったもの上位5位までの，出力の値とクラス名を表します．
ネットワークの出力層の活性化関数は softmax ですので，出力の値は0から1でかつ1000子の出力の値すべての和が 1 となっています（一般のロジスティック回帰モデルの説明参照）．
これらの値は，そのクラスと判定することの「確信度」と解釈できます．

---
#### 実験2

自分で用意した画像でも実験してみましょう．



(1) まずは，ウェブ等で適当な画像を探して手元に保存しましょう．JPEG（拡張子は `.jpg` や `.jpeg` 等）やPNG（`.png`）等の形式のものが扱えます．ファイル名が長かったり日本語を含んでいる場合は，短い名前に変更しておくのがよいです．

(2) 以下のセルを実行して，ファイルをアップロードします．

In [ ]:
# Colab へファイルをアップロード
from google.colab import files
rv = files.upload()

# ファイル一覧
! ls

(3) 以下のセルの1行目のファイル名を上記でアップロードしたものに変えて実行しましょう．アルファベット大文字小文字の区別もありますので注意．うまくいけば画像が表示されるはずです．

In [ ]:
fn = 'hoge.jpg' # ファイル名を変更しましょう

myimg = Image.open(fn)
if myimg.mode != 'RGB':
    myimg = myimg.convert('RGB')
fig, ax = plt.subplots(1)
ax.imshow(myimg)
ax.axis('off')
plt.show()

(4) 以下のセルを実行すれば結果が表示されます．

In [ ]:
# VGG16 ネットワークに入力して出力を得る
X = torch.unsqueeze(preprocess(myimg), axis=0)
Y = model(X)
Z = torch.nn.functional.softmax(Y, dim=1)
P = Z[0].detach().numpy()

# 出力の値の大きかった方から5つを表示
for i, ik in enumerate(np.argsort(-P)[:5]):
    print(f'rank{i+1}: {P[ik]:.6f} {labels[ik]}')


実験1は動物の例ばかりでしたが，1000のクラスの中にはそれ以外にも様々なものがあります（「人間」を表すクラスはありません）．いろいろ試してみてください．面白い／不思議な結果が得られたら takataka に見せてくれると喜びます．
